# TranslationDataset & DataManager Walkthrough

In [1]:
from pathlib import Path
import json
import sys
import yaml

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "config.yaml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

with open(PROJECT_ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

DATA_DIR   = PROJECT_ROOT / config["paths"]["data_dir"]
TRAIN_PATH = DATA_DIR / config["data"]["train"]
VAL_PATH   = DATA_DIR / config["data"]["val"]
TEST_PATH  = DATA_DIR / config["data"]["test"]


## 1. What does the raw JSON look like?

In [2]:
with open(TRAIN_PATH, encoding="utf-8") as f:
    raw = json.load(f)

print(type(raw), len(raw))
raw[0]

<class 'list'> 13935


{'source': 'The well will be drilled in the south-eastern part of the licence, which is located in the central North Sea.',
 'target': 'Brønnen skal borast i den søraustlege delen av løyvet, som ligg i midtre del av Nordsjøen.',
 'source_lang': 'en',
 'target_lang': 'no'}

## 2. TranslationSample — wrap one dict into an object

In [3]:
from scripts.data.dataset import TranslationSample

item = raw[0]
sample = TranslationSample(
    source=item["source"],
    target=item["target"],
    metadata=item.get("metadata"),
)

print(sample.source)
print(sample.target)
print(sample.metadata)
print(sample.to_dict())   # None metadata becomes {}

The well will be drilled in the south-eastern part of the licence, which is located in the central North Sea.
Brønnen skal borast i den søraustlege delen av løyvet, som ligg i midtre del av Nordsjøen.
None
{'source': 'The well will be drilled in the south-eastern part of the licence, which is located in the central North Sea.', 'target': 'Brønnen skal borast i den søraustlege delen av løyvet, som ligg i midtre del av Nordsjøen.', 'metadata': {}}


## 3. TranslationDataset.from_json() — load the whole file at once

In [4]:
from scripts.data.dataset import TranslationDataset

train_ds = TranslationDataset.from_json(
    TRAIN_PATH,
    src_lang=config["model"]["src_lang"],
    tgt_lang=config["model"]["tgt_lang"],
)

print(len(train_ds), train_ds.src_lang, train_ds.tgt_lang)
train_ds[0]   # calls __getitem__ → sample.to_dict()

13935 eng_Latn nob_Latn


{'source': 'The well will be drilled in the south-eastern part of the licence, which is located in the central North Sea.',
 'target': 'Brønnen skal borast i den søraustlege delen av løyvet, som ligg i midtre del av Nordsjøen.',
 'metadata': {}}

## 4. get_statistics() — sentence length distribution

In [5]:
# whitespace-based word counts, not tokenizer lengths
train_ds.get_statistics()

{'total': 13935,
 'avg_source_length': 17.8599928238249,
 'avg_target_length': 14.791101542877646,
 'max_source_length': 123,
 'max_target_length': 123}

## 5. subset() — reproducible random sample

In [6]:
small = train_ds.subset(5, seed=42)

for i, s in enumerate(small.samples, 1):
    print(f"{i}. {s.source}")
    print(f"   {s.target}\n")

1. +47 87 61 00
   51 87 61 00.

2. Total oil production is slightly above the NPD’s prognosis for the year.
   Total oljeproduksjon er litt over OD sin prognose for 2012.

3. The stand features, among others, a core sample from the very first well that proved oil on the Norwegian shelf in 1967.
   På standen er det blant annet utstilt en kjerneprøve fra den aller første brønnen som påviste olje på norsk sokkel i 1967.

4. It encountered the Springar formation with reservoir rocks and reservoir quality as expected, but the well is dry.
   Den påtraff Springarformasjonen med reservoarbergarter og reservoarkvalitet som forventet, men brønnen er tørr.

5. Analyses of the core material have resulted in a number of scientific publications and reports.
   Analyser av kjernematerialet har resultert i en rekke vitenskapelige publikasjoner og rapporter.



In [7]:
# same seed → same result every time
a = train_ds.subset(5, seed=42)
b = train_ds.subset(5, seed=42)
print(a[0] == b[0])

True


## 6. to_json() — save dataset back to disk

In [ ]:
out = PROJECT_ROOT / "tmp_check.json"
small.to_json(out)

with open(out, encoding="utf-8") as f:
    saved = json.load(f)

print(len(saved), saved[0])
out.unlink()  # clean up

## 7. DataManager — load all three splits in one call

In [8]:
from scripts.data.data_loader import DataManager

dm = DataManager(config)
train, val, test = dm.load_splits()

{"train": len(train), "val": len(val), "test": len(test)}

{'train': 13935, 'val': 1737, 'test': 1742}

In [9]:
# subset only the training split
train_small, val, test = dm.load_splits(train_subset_size=100)
print(len(train_small), len(val), len(test))

100 1737 1742


## 8. reverse=True — swap EN↔NO

In [10]:
rev, _, _ = dm.load_splits(train_subset_size=1, reverse=True)

print("lang:", rev.src_lang, "→", rev.tgt_lang)
print("original:", train[0])
print("reversed:", rev[0])

lang: nob_Latn → eng_Latn
original: {'source': 'The well will be drilled in the south-eastern part of the licence, which is located in the central North Sea.', 'target': 'Brønnen skal borast i den søraustlege delen av løyvet, som ligg i midtre del av Nordsjøen.', 'metadata': {}}
reversed: {'source': 'Den påtraff Springarformasjonen med reservoarbergarter og reservoarkvalitet som forventet, men brønnen er tørr.', 'target': 'It encountered the Springar formation with reservoir rocks and reservoir quality as expected, but the well is dry.', 'metadata': {}}


config.yaml
    │
    ▼
DataManager.load_splits()
    │  in:  train.json  →  [{"source": "The valve shall be closed.", "target": "Ventilen skal stenges."},  ...]
    │  out: TranslationDataset  →  dataset[0] = {"source": "...", "target": "...", "metadata": {}}
    │
    ▼
BaseTrainer.tokenize()
    │  in:  {"source": "The valve shall be closed.", "target": "Ventilen skal stenges."}
    │  out: input_ids      = [128022, 1636, 23983, 953, 128, 2]        ← English tokens
    │       attention_mask = [1, 1, 1, 1, 1, 1]
    │       labels         = [128008, 1110, 6765, 3, -100, -100]       ← Norwegian tokens, pad→-100
    │
    ▼
DataCollatorForSeq2Seq
    │  in:  batch of sequences with different lengths
    │       seq1: [128022, 1636, 23983, 2]
    │       seq2: [128022, 451, 2]
    │  out: padded tensors of same length
    │       [[128022, 1636, 23983,     2],
    │        [128022,   451,     2, 1]  ]   ← 1 = PAD token
    │
    ▼
NLLB-600M (Encoder-Decoder)
    │  in:  padded input_ids tensor  [batch, seq_len]
    │  out: training   →  loss = 2.34  (cross-entropy over labels)
    │       inference  →  generated token IDs = [128008, 1110, 6765, 953, 2]
    │
    ▼
tokenizer.batch_decode()
    │  in:  [128008, 1110, 6765, 953, 2]
    │  out: "Ventilen skal stenges."
    │
    ▼
compute_metrics()
    │  in:  prediction = "Ventilen skal stenges."
    │       reference  = "Ventilen skal stenges."
    │  out: BLEU=42.3,  chrF=61.7,  COMET=0.84

config.yaml
    │
    ▼
Load NLLB Model & Tokenizer
    │
    ▼
DataManager.load_splits()
    │
    ▼
TranslationDataset
(train / val / test)
    │
    ▼
BaseTrainer.tokenize()
    │
    ▼
DataCollatorForSeq2Seq
    │
    ▼
Seq2SeqTrainer
    │
    ├─ TrainingArguments
    ├─ Optimizer
    ├─ Scheduler
    └─ compute_metrics
    │
    ▼
NLLB-600M
(Encoder–Decoder)
    │
 ┌──┴──────────────┐
 │                 │
 ▼                 ▼
Training       Evaluation
 │                 │
loss        model.generate()
 │                 │
 ▼                 ▼
checkpoint   batch_decode()
                   │
                   ▼
           BLEU / chrF / COMET
                   │
                   ▼
           Prediction Outputs

config.yaml
        │  pretrained: facebook/nllb-200-distilled-600M
        │  src_lang: eng_Latn
        │  tgt_lang: nob_Latn
        │
        │  batch_size: 4
        │  gradient_accumulation_steps: 4
        │  effective batch size = 16
        │
        │  learning_rate = 5e-4
        │  warmup_steps = 100
        │  epochs = 3
        │  eval_steps = 200
        │
        ▼
load_splits()
        │
        ├─ train.json → 13,935 pairs
        ├─ val.json   →  1,737 pairs
        └─ test.json  →  1,742 pairs
        │
        ▼
train / val / test
        │
        │ Example sample:
        │ {
        │   "source":
        │   "The valve shall be closed before maintenance.",
        │
        │   "target":
        │   "Ventilen skal stenges før vedlikehold."
        │ }
        │
        ▼
AutoTokenizer
        │
        │ model:
        │ facebook/nllb-200-distilled-600M
        │
        │ src_lang = eng_Latn
        │ tgt_lang = nob_Latn
        │
        │ vocabulary size:
        │ 256,206 tokens
        │
        ▼
tokenize()
(dataset.map(batched=True) over all 13,935 training samples)
        │
        │ source:
        │ "The valve shall be closed before maintenance."
        │
        ├─ input_ids
        │  [128022, 1636, 23983, 953, 764, 7396, 2]
        │
        ├─ attention_mask
        │  [1, 1, 1, 1, 1, 1, 1]
        │
        │ target:
        │ "Ventilen skal stenges før vedlikehold."
        │
        └─ labels
           [128008, 1110, 6765, 953, 764, 2]
        │
        │ padding=False
        │ (no padding applied at tokenization stage)
        │
        ▼
DataCollatorForSeq2Seq
(dynamic padding at training time)
        │
        │ batch size = 4
        │ padded to longest sequence in current batch
        │
        │ input_ids:
        │
        │ [[128022, 1636, 23983, 953, 764, 7396,    2],
        │  [128022,  451,   349,   2,   1,    1,    1],
        │  [128022,  890,  2341, 567, 234,    2,    1],
        │  [128022,  123,     2,   1,   1,    1,    1]]
        │
        │ PAD token id = 1
        │
        │ labels:
        │
        │ [[128008,1110,6765,953,764,2,-100],
        │  [...],
        │  [...],
        │  [...]]
        │
        │ -100 positions ignored by CrossEntropyLoss
        │
        ▼
Batch Tensors
        │
        ├─ input_ids       [4, L]
        ├─ attention_mask  [4, L]
        └─ labels          [4, L]
        │
        │ L = longest sequence length
        │ in current batch
        │
        ▼
Seq2SeqTrainer
        │
        │ steps per epoch
        │ = 13,935 / (4 × 4)
        │ ≈ 870
        │
        │ total training steps
        │ = 870 × 3
        │ ≈ 2,610
        │
        │ validation frequency
        │ = every 200 steps
        │
        │ total evaluation runs
        │ ≈ 13
        │
        ├─ AdamW
        │    lr = 5e-4
        │    weight_decay = 0.01
        │    max_grad_norm = 1.0
        │
        ├─ LR Scheduler
        │    100 warmup steps
        │    → linear decay
        │
        ├─ EarlyStopping
        │    patience = 3 evaluations
        │    (≈ 600 training steps
        │     when eval_steps = 200)
        │
        └─ compute_metrics()
             runs on 1,737 validation samples
             every 200 steps
        │
        ▼
NLLB-600M
(+ LoRA adapters)
        │
        │ LoRA configuration
        │
        │ r = 16
        │ α = 32
        │ dropout = 0.1
        │
        │ target modules:
        │ q_proj
        │ k_proj
        │ v_proj
        │ out_proj
        │
        │ only LoRA weights updated
        │ (~0.5% of total parameters)
        │
        ┌──────┴──────────────────────────────┐
        │                                     │
        ▼                                     ▼
Training                              Validation
        │                                     │
        │ forward pass                        │ model.generate()
        │                                     │
        │ logits                              │ forced_bos_token_id
        │ shape ≈ [4, L, 256206]              │ = nob_Latn
        │                                     │
        │ cross-entropy loss                  │ num_beams = 5
        │                                     │ max_length = 128
        │ backward pass                       │
        │                                     │ generated_ids:
        │ update LoRA weights                 │
        │                                     │ [128008,
        │                                     │  1110,
        │                                     │  6765,
        │                                     │  953,
        │                                     │  764,
        │                                     │  2]
        │                                     │
        ▼                                     ▼
checkpoint saved                      batch_decode()
        │                                     │
        │                                     │
        │                                     ▼
        │                             "Ventilen skal stenges
        │                              før vedlikehold."
        │                                     │
        │                                     ▼
        │                              BLEU / chrF
        │
        │                              Example progression:
        │                              step 200:
        │                                BLEU = 31.2
        │                                chrF = 54.1
        │
        │                              step 400:
        │                                BLEU = 38.7
        │                                chrF = 59.3
        │
        │                              step 600:
        │                                BLEU = 41.5
        │                                chrF = 61.2
        │
        │                              (illustrative example)
        │
        ▼
Best checkpoint selected
(metric_for_best_model = BLEU)
        │
        │ save_total_limit = 2
        │ keep best 2 checkpoints
        │
        ▼
load_best_model_at_end=True
        │
        │ best checkpoint automatically
        │ reloaded after training
        │
        ▼
trainer.evaluate()
(on validation set)
        │
        ▼
Final Validation Metrics
        │
        ├─ BLEU
        ├─ chrF
        └─ loss
        │
        ▼
generate_predictions()
        │
        │ test set:
        │ 1,742 samples
        │
        │ batch_size = 8
        │
        │ number of batches:
        │ ceil(1742 / 8)
        │ = 218
        │
        ▼
Predictions
        │
        ├─ "Ventilen skal stenges før vedlikehold."
        ├─ ...
        └─ ...
        │
        ▼
Test Evaluation
        │
        ├─ BLEU  = 42.3
        ├─ chrF  = 61.7
        └─ COMET = 0.84
        │
        │ COMET evaluated only on final test set
        │ because it is substantially slower than
        │ BLEU and chrF during validation
        │
        ▼
outputs/
        │
        ├─ checkpoints/
        ├─ predictions.json
        ├─ metrics.json
        └─ results.json